# Beach Wave Collision Simulation
## Physics and Mathematical Formulation (Corrected Geometry)

This presentation extracts the underlying physics and mathematical framework from the 2D scalar wave equation simulation of colliding water blades, corrected to match the actual visual output.

---

## 1. Physical Setup and Geometry

The simulation models the interaction of two water blades traveling toward each other, mimicking the **swash/backwash interaction** observed on gently sloping beaches. Their collision generates a lateral wave.

**Domain Geometry (as plotted):**
*   **Longshore direction ($x$):** $x \in [-L_x/2, L_x/2]$ (Horizontal axis, periodic boundary conditions)
*   **Cross-shore direction ($y$):** $y \in [-L_y/2, L_y/2]$ (Vertical axis)

**Blade Trajectories:**
*   **Blade 1:** Starts at $y = +Y_1$, traveling in the $-y$ direction (parallel to the $x$-axis, angle $\theta_1 = 0$).
*   **Blade 2:** Starts at $y = -Y_2$, traveling in the $+y$ direction, but slightly tilted at an angle $\theta_2 = \alpha$.

*(Note: The original code comments described the blades traveling along the x-axis. However, due to an array indexing mismatch between NumPy's `meshgrid` and the solver's internal `ij` indexing, the physical x and y axes are transposed in the final plot.)*

---

## 2. The Governing Wave Equation

The system is governed by a damped 2D scalar wave equation, implemented using a **pseudodifferential operator** ($\Psi_{\text{op}}$):

$$
\frac{\partial^2 u}{\partial t^2} = -\Psi_{\text{op}}(a) u - \Gamma \frac{\partial u}{\partial t}
$$

Where:
*   $u(x,y,t)$ is the wave elevation (scalar field).
*   $\Psi_{\text{op}}(a)$ is the pseudodifferential operator defined by its principal symbol $a(x, y, \xi, \eta)$.
*   $\Gamma$ is the bottom friction coefficient (damping term).

---

## 3. The Principal Symbol

In the framework of microlocal analysis, the spatial differential operator is replaced by its **principal symbol** $a(x, y, \xi, \eta)$ in the Fourier domain (where $\xi$ and $\eta$ are the spatial frequencies conjugate to $x$ and $y$). 

The total symbol is composed of three distinct physical phenomena:

$$
a(\xi, \eta) = \underbrace{C^2 \left( A_{xx} \xi^2 + 2 A_{xy} \xi \eta + A_{yy} \eta^2 \right)}_{\text{Wave Propagation \& Anisotropy}} + \underbrace{F_{\text{Coriolis}} \xi \eta}_{\text{Coriolis Deflection}} + \underbrace{F_{\text{Vorticity}} (x \eta - y \xi)}_{\text{Background Vorticity}}
$$

*   $C^2$: The base wave speed squared.
*   $A_{xx}, A_{yy}, A_{xy}$: Components of the anisotropy matrix.
*   $F_{\text{Coriolis}}$: The Coriolis parameter ($f = 2\Omega \sin(\text{latitude})$).
*   $F_{\text{Vorticity}}$: Background vorticity parameter.

---

## 4. Physical Effects Encoded in the Math

The mathematical formulation elegantly captures several complex fluid dynamics effects:

1.  **Anisotropy:** The matrix coefficients $A_{xx}, A_{yy}, A_{xy}$ allow the wave to propagate at different speeds in the longshore ($x$) vs. cross-shore ($y$) directions. If $A_{yy} > A_{xx}$, cross-shore propagation is faster. A non-zero $A_{xy}$ couples the $x$ and $y$ directions.
2.  **Coriolis Effect:** The term $F_{\text{Coriolis}} \xi \eta$ introduces a cross-coupling in Fourier space that physically deflects wave energy sideways (e.g., to the right in the Northern Hemisphere).
3.  **Background Vorticity (Rotation):** The term $F_{\text{Vorticity}} (x \eta - y \xi)$ introduces a rotational component to the wave dynamics. Mathematically, the expression $(x \eta - y \xi)$ is the principal symbol for the angular momentum operator (or rotation generator) $x \partial_y - y \partial_x$ in physical space. Physically, this simulates a background rotational flow—such as a large-scale oceanic eddy or topographic vorticity—that advects and rotates the propagating wave field around the center of the domain.
4.  **Bottom Friction:** The term $-\Gamma \frac{\partial u}{\partial t}$ acts as a linear drag, dissipating energy over time so the blades decay realistically as they travel and collide.

---

## 5. Initial Conditions: The Water Blades

To simulate the physical "blades" of water, the initial surface elevation $u(x,y,0)$ is constructed using a combination of a **Gaussian envelope** and a **smoothed Heaviside step function (sigmoid)**.

**Blade Profile $B(x,y)$:**
$$
B(x,y) = \exp\left(-\frac{d^2}{2\sigma^2}\right) \cdot \frac{1}{1 + \exp(k_h \cdot s \cdot d)}
$$

Where:
*   $d = \cos(\theta)(y - y_c) + \sin(\theta)x$ is the signed distance from the blade center (adjusted for y-direction propagation).
*   $\sigma = \lambda / \text{width\_ratio}$ controls the thickness of the blade.
*   $k_h$ controls the steepness (sharpness) of the leading edge.
*   $s \in \{-1, +1\}$ determines the direction the blade is facing.

**Initial Velocity $\partial_t u(x,y,0)$:**
To ensure the blades propagate towards each other at the correct physical speed (based on WKB approximation principles), the initial velocity is set proportional to the spatial derivative of the blade profile:

$$
v(x,y) = -s \cdot c \cdot \frac{\partial B}{\partial d} \cdot \text{SpeedFactor}
$$

Where $c = \sqrt{C^2}$ is the local wave speed.

---

## 6. Key Simulation Parameters

| Parameter | Symbol | Description |
| :--- | :---: | :--- |
| **Wave Speed** | $C^2$ | Constant wave speed squared (m²/s²). |
| **Wavelength** | $\lambda$ | Characteristic wavelength controlling blade width (m). |
| **Blade Width** | $\sigma$ | Standard deviation of the Gaussian profile ($\sigma = \lambda / \text{ratio}$). |
| **Tilt Angle** | $\alpha$ | Angle between the two colliding blades (degrees). |
| **Friction** | $\Gamma$ | Bottom friction coefficient (s⁻¹); dictates energy dissipation. |
| **Coriolis** | $F_{\text{Coriolis}}$ | Coriolis parameter (s⁻¹); dictates lateral deflection. |
| **Vorticity** | $F_{\text{Vorticity}}$ | Background vorticity parameter (s⁻¹); induces rotational advection. |
| **Anisotropy** | $A_{xx}, A_{yy}$ | Weights for longshore and cross-shore propagation speeds. |

# Implementation
## 0. Imports

In [ ]:
from solver import *
import sympy as sp
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import HTML

## 1. Physical and simulation parameters

In [ ]:
# ── Wave speed ──
C_SQUARED = 1.0        # constant c² (m²/s²); replace with G*ALPHA*(x+X0) for variable speed

# ── Wave / blade properties ──
LAMBDA             = 1.0   # characteristic wavelength controlling blade width (m)
BLADE_WIDTH_RATIO  = 1.0   # sigma = LAMBDA / BLADE_WIDTH_RATIO  (larger → narrower)
HEAVISIDE_SHARPNESS = 1.0 # steepness of the blade leading edge  (larger → sharper front)
SPEED_FACTOR       = 0.5   # initial velocity scale  (1.0 = full WKB, <1 = slower)

# ── Blade geometry ──
ALPHA_DEG          = 0.0   # tilt angle of blade 2 relative to blade 1 (degrees)
ALPHA_R            = np.radians(ALPHA_DEG)
BLADE1_ANGLE       = 0.0   # blade 1: parallel to y-axis, travels in -x
BLADE2_ANGLE       = ALPHA_R  # blade 2: tilted by ALPHA_R, travels in +x

# ── Anisotropy matrix  A = [[A_XX, A_XY], [A_XY, A_YY]] ──
# symbol_wave = C² · (A_XX·ξ² + 2·A_XY·ξη + A_YY·η²)
# A_XX > A_YY : faster cross-shore than longshore propagation
# A_XY ≠ 0   : couples x and y directions
A_XX = 1.0   # cross-shore weight
A_YY = 0.5  # longshore weight  (set < 1 to slow down longshore propagation)
A_XY = 0.4   # off-diagonal coupling (0 = principal axes aligned with x, y)

# ── Dissipation and Coriolis ──
GAMMA      = 0.0    # bottom friction (s⁻¹); 0 = no decay, 0.05 = moderate decay
F_CORIOLIS = 0.0   # Coriolis parameter (s⁻¹); 0 = off, >0 = Northern hemisphere
F_VORTICITY = 0.5

# ── Velocity balance between the two blades ──
# Compensates for amplitude differences between straight and angled blades.
VELOCITY_SCALE_BLADE2 = 1


## 2. Grid setup

In [ ]:
Lx, Ly   = 10.0, 14.0
# Nx, Ny = 32, 32 
# Nx, Ny = 64, 64    
Nx, Ny = 128, 128    

# Lt, Nt   = 20.0, 400
# Lt, Nt   = 30.0, 600
# Lt, Nt   = 40.0, 800
# Lt, Nt   = 50.0, 1000
Lt, Nt   = 60.0, 1200

n_frames = 400

SIGMA = LAMBDA / BLADE_WIDTH_RATIO
Y1 =  Lx / 2.0 - SIGMA
Y2 = -Lx / 2.0 + SIGMA

xs_1d = np.linspace(-Lx/2, Lx/2, Nx)
ys_1d = np.linspace(-Ly/2, Ly/2, Ny)

# CRITICAL FIX: psipy internally uses indexing='ij' (axis 0 = x, axis 1 = y).
# We MUST match this convention so the solver receives the correct array orientation.
xx, yy = np.meshgrid(xs_1d, ys_1d)   # shape (Nx, Ny)

## 3. SymPy symbols and principal symbol

In [ ]:
x, y, t  = sp.symbols('x y t', real=True)
xi, eta  = sp.symbols('xi eta', real=True)
u_func   = Function('u')
u        = u_func(t, x, y)

# Full principal symbol:
#   a(ξ) = C²·(A_XX·ξ² + 2·A_XY·ξη + A_YY·η²)   ← wave propagation + anisotropy
#          + F_CORIOLIS·ξη                          ← Coriolis deflection
symbol_wave     = C_SQUARED * (A_XX * xi**2 + 2*A_XY * xi*eta + A_YY * eta**2)
symbol_coriolis = F_CORIOLIS * xi * eta
symbol_vorticity = F_VORTICITY * (x* eta - y * xi)
symbol_num      = symbol_wave + symbol_coriolis + symbol_vorticity

print("Principal symbol:")
print("  a(ξ) =", symbol_num)


## 4. Wave equation

In [ ]:
#
#   ∂²u/∂t² = -psiOp(a(ξ), u) - GAMMA·∂u/∂t
#
#   Term 1: -psiOp(a(ξ), u)  — wave propagation, anisotropy, Coriolis (in symbol)
#   Term 2: -GAMMA·∂u/∂t     — bottom friction (energy dissipation, not in symbol
#                               because it is a zeroth-order term in space)

gamma    = sp.Symbol('gamma', positive=True)
equation = sp.Eq(
    diff(u, t, 2),
    -psiOp(symbol_num, u) - gamma * diff(u, t)
)
equation_num = equation.subs({gamma: GAMMA})

print("Equation:")
print(f"  ∂²u/∂t² = -psiOp(a(ξ), u) - {GAMMA}·∂u/∂t")


## 5. Initial conditions

In [ ]:
def _blade(xx, yy, x_center, sign, angle=0.0):
    """
    Single water blade localised around x_center.
    Grid convention: indexing='xy'  →  axis 0 = y, axis 1 = x.
    """
    nx, ny = np.cos(angle), np.sin(angle)
    # x varies along axis=1, y along axis=0 → formula unchanged
    dist      = nx * (xx - x_center) + ny * yy
    sigma     = LAMBDA / BLADE_WIDTH_RATIO
    gauss     = np.exp(-(dist**2) / (2.0 * sigma**2))
    k_h       = HEAVISIDE_SHARPNESS / sigma
    arg       = np.clip(k_h * sign * dist, -500, 500)
    heaviside = 1.0 / (1.0 + np.exp(arg))
    return gauss * heaviside


def _blade_velocity(xx, yy, x_center, sign, angle=0.0):
    nx, ny   = np.cos(angle), np.sin(angle)
    dist     = nx * (xx - x_center) + ny * yy
    sigma    = LAMBDA / BLADE_WIDTH_RATIO
    k_h      = HEAVISIDE_SHARPNESS / sigma
    arg      = np.clip(k_h * sign * dist, -500, 500)
    gauss    = np.exp(-(dist**2) / (2.0 * sigma**2))
    sigmoid  = 1.0 / (1.0 + np.exp(arg))
    
    d_gauss   = -(dist / sigma**2) * gauss
    # FIX 1: Added the missing negative sign here
    d_sigmoid = - sign * k_h * sigmoid * (1.0 - sigmoid) 
    
    d_blade   = d_gauss * sigmoid + gauss * d_sigmoid
    c_local   = np.sqrt(max(C_SQUARED, 1e-6))
    return -sign * c_local * d_blade * SPEED_FACTOR


def initial_condition_b(xx, yy):
    return (_blade(xx, yy, Y1, sign=-1, angle=BLADE1_ANGLE) +
            _blade(xx, yy, Y2, sign=+1, angle=BLADE2_ANGLE))

def initial_velocity_b(xx, yy):
    vel1 = _blade_velocity(xx, yy, Y1, sign=-1, angle=BLADE1_ANGLE)
    vel2 = _blade_velocity(xx, yy, Y2, sign=+1, angle=BLADE2_ANGLE)
    # FIX 2: Changed '-' to '+' so Blade 2 travels in the +x direction
    return vel1 + VELOCITY_SCALE_BLADE2 * vel2 

## 6. Solver setup

In [ ]:
solver = PDESolver(equation_num)

solver.setup(
    Lx=Lx, Ly=Ly,
    Nx=Nx, Ny=Ny,
    Lt=Lt, Nt=Nt,
    boundary_condition='periodic',
    initial_condition=initial_condition_b,
    initial_velocity=initial_velocity_b,
    n_frames=n_frames,
    plot=True,
)


## 7. Solve

In [ ]:
frames = solver.solve()


## 8. Visualization

In [ ]:
# Raise the animation size limit to allow large frame counts at high resolution
plt.rcParams['animation.embed_limit'] = 2**128

ani = solver.animate(
    component='real',
    overlay=None,
    mode='imshow',
)

HTML(ani.to_jshtml())


In [ ]:
ani = solver.animate(
    component='real',
    overlay=None,
    mode='surface',
)

HTML(ani.to_jshtml())